# St. Louis Crime Analysis for Uber Driver Safety
## Data Science Final Project - Team Review Document

---

### **Project Objectives**

This project aims to create a comprehensive safety and demand analysis system for rideshare drivers (specifically Uber drivers) operating in St. Louis, Missouri. By analyzing historical crime data and geographic information, we provide actionable insights to help drivers:

1. **Identify safe neighborhoods and time periods** for operation
2. **Avoid high-risk areas** with violent crime incidents
3. **Maximize earnings** by identifying high-demand areas that are also safe
4. **Optimize work schedules** based on temporal crime patterns

---

### **Data Sources**

1. **St. Louis Crime Dataset**: `February2026.csv`
   - Contains incident reports with timestamps, locations, crime types, and severity
   - Fields include: IncidentDate, OccurredFromTime, Neighborhood, CrimeAgainst, Latitude, Longitude

2. **Geographic Data API**: Zippopotam.us API
   - Provides zip code to coordinate mapping for St. Louis area
   - Used to associate crime incidents with specific zip codes
3. **Traffic & Road Conditions Data**: Simulated dataset
   - **Metrics**: Traffic flow index, road quality scores, slope grades, weather risk zones
   - **Purpose**: Demonstrate multi-factor recommendation system
   - **Status**:  **Simulated for demonstration** (not real API data)
   - **Note**: In production deployment, would integrate with:
     - TomTom/Google Maps Traffic API (free) : Traffic Data
     - OpenStreetMap road quality data (free) : Road Quality
     - USGS Elevation API (free) : Topography
     - NOAA Weather Service (free):Weathe
---

### **Methodology**

1. **Data Collection & Integration**
   - Fetch geographic data from public API
   - Load and clean crime dataset
   - Merge datasets using nearest-neighbor geographic matching

2. **Feature Engineering**
   - Extract temporal features (hour, day of week, month)
   - Calculate safety scores per neighborhood
   - Identify violent vs. non-violent crimes

3. **Analysis & Modeling**
   - Spatial analysis: crime distribution by neighborhood and zip code
   - Temporal analysis: hourly and daily crime patterns
   - Risk assessment: combining safety and activity metrics

4. **Visualization & Recommendations**
   - Static visualizations (heatmaps, time series, geographic scatter plots)
   - Interactive dashboards using Plotly
   - Actionable recommendation system

---

### **Expected Outcomes**

- **Safety Score Cards** for each neighborhood and time period
- **Interactive Maps** showing crime hotspots and safe zones
- **Smart Recommendation Engine** that suggests optimal work locations and times
- **Risk Alerts** for high-danger combinations of location + time



# Data Engineering Process Documentation
## St. Louis Crime Analysis Project - Standardized Methodology

---

## Executive Summary

This document provides a comprehensive overview of all data engineering processes, techniques, and methodologies we applied in the St. Louis Crime Analysis for Uber Driver Safety project.

---

# 1. DATA ACQUISITION & INGESTION

## 1.1 External API Data Collection

**Objective:** Acquire geographic reference data for St. Louis zip codes

**Source:** Zippopotam.us Public API  
**Endpoint:** `https://api.zippopotam.us/us/{zip_code}`  

**Implementation Details:**
- **Technology Stack:** Python `requests` library
- **Target Data:** 5 St. Louis zip codes (63101-63105)
- **Error Handling:** Try-except blocks with timeout configuration (10 seconds)

**Data Extracted:**
- Place name (neighborhood/city identifier)
- Geographic coordinates (latitude, longitude)
- Zip code (5-digit postal code)
- State information

---

## 1.2 Automated CSV Data Ingestion from Web Source

**Objective:** Programmatically fetch historical crime incident records from official source

**Source:** St. Louis Metropolitan Police Department (SLMPD)  
**URL:** `https://slmpd.org/wp-content/uploads/2026/03/February2026.csv`  
**Method:** Automated HTTP GET request via Python `requests` library  


**Automation Strategy:**
- **Programmatic Download:** Code automatically fetches data from source URL
- **Timeout Handling:** 30-second timeout to prevent hanging connections
- **File Persistence:** Downloaded content saved to workspace for processing


**Why Automated Ingestion:**
- **Reliability & Scale: Enables scheduled, hands-free updates that mirror professional production pipelines.
- **Efficiency: Keeps Git repositories lightweight while ensuring the entire process is repeatable and auditable.

---

## 1.3 Simulated Traffic & Road Conditions Data

**Objective:** Demonstrate multi-factor recommendation system (prototype only)

**Why Simulated?**
- **Budget & Access Constraints: Premium APIs (like Google/TomTom) require high monthly fees and credit card verification, which were not feasible for this student project.

-Proof of Concept: Using simulated data effectively demonstrates the system's ability to handle multi-factor analysis and its readiness for future production integration.

**Data Generated:**
### Traffic Flow Metrics
- **Traffic Flow Index**: 0-100 scale
- **Congestion Level**: Low/Medium/High
- **Average Speed**: mph values

### Road Quality Metrics
- **Road Quality Score**: 0-100 scale
- **Surface Condition**: Excellent/Good/Fair/Poor
- **Last Maintenance Date**: YYYY-MM format

### Topographic Data
- **Average Slope**: Percentage (%)
- **Max Slope**: Maximum grade in area
- **Winter Risk Level**: Low/Medium/High

### Weather Risk Zones
- **Flood Risk**: Low/Medium/High
- **Ice Accumulation Risk**: Based on elevation + slope
- **Snow Drift Risk**: Open area exposure

# 2. DATA CLEANING & PREPROCESSING

## 2.1 Data Type Conversion

**Objective:** Ensure data types match expected formats for analysis

**Transformations Applied:**
- **Geospatial Data: Used pd.to_numeric on Latitude and Longitude with errors='coerce' to enable distance calculations.

- **Temporal Data: Converted IncidentDate and OccurredFromTime into standard datetime objects for time-based filtering.

- **Quality Impact: Ensures data consistency and enables advanced feature extraction (e.g., hourly/daily crime trends).

## 2.2 Missing Data Handling

**Strategy:** Record removal for critical fields

**Implementation:**
```python
df_crime.dropna(subset=['Latitude', 'Longitude'])
```
**Rationale:**
- Geographic coordinates are **essential** for spatial analysis
- Missing coordinates cannot be reliably imputed
- Removal ensures data quality for location-based features

**Impact Metrics:**
- Records removed: Tracked and reported
- Retention rate: Calculated as percentage
- Validation: Ensures majority of data retained (>90% typical)

---

## 2.3 Data Validation

**Coordinate Validation:**
- Verify latitude/longitude within St. Louis bounds
- Check for outliers using basic statistics
- Ensure non-null values after conversion

**Temporal Validation:**
- Verify date range consistency
- Check for future dates (data quality issue)
- Validate time format (0-23 hours)

---

# 3. FEATURE ENGINEERING

## 3.1 Temporal Feature Extraction

**Objective:** Extract time-based features for temporal pattern analysis

**Features Created:**

| Feature | Extraction Method | Purpose |
|---------|-------------------|----------|
| `Hour` | `dt.hour` | Identify hourly crime patterns |
| `DayOfWeek` | `dt.day_name()` | Analyze weekly cycles |
| `Month` | `dt.month` | Track monthly trends |
| `Year` | `dt.year` | Multi-year comparison |

**Technical Implementation:**
```python
df['Hour'] = df['OccurredFromTime'].dt.hour
df['DayOfWeek'] = df['IncidentDate'].dt.day_name()
df['Month'] = df['IncidentDate'].dt.month
df['Year'] = df['IncidentDate'].dt.year
```

**Use Cases:**
- Hourly crime heatmaps
- Day-of-week risk assessment
- Seasonal pattern identification
- Time-based recommendation filtering

---

## 3.2 Geospatial Feature Engineering

**Objective:** Assign zip codes to crime incidents using spatial relationships

**Algorithm:** Nearest-Neighbor Geographic Matching

**Mathematical Foundation:**
```
Euclidean Distance = √[(lat₁ - lat₂)² + (lon₁ - lon₂)²]

Assigned Zip Code = argmin(distances)
```

**Implementation:**
```python
def find_nearest_zipcode(lat, lon, zip_df):
    distances = np.sqrt(
        (zip_df['latitude'] - lat)**2 + 
        (zip_df['longitude'] - lon)**2
    )
    nearest_idx = distances.idxmin()
    return zip_df.loc[nearest_idx, 'zip_code']
```

**Complexity Analysis:**
- Time Complexity: O(n × m) where n = crime records, m = zip codes
- Space Complexity: O(n) for storing zip code assignments

**Assumptions & Limitations:**
- Zip code coordinates represent centroids
- Euclidean distance sufficient for small areas (St. Louis)
- Boundary edge cases acceptable for analysis granularity

**Validation:**
- Distribution check: All 5 zip codes represented
- No assignments to distant/invalid zip codes
- Reasonable distribution across target zones

---

## 3.3 Derived Metrics & Indicators

### Safety Score Calculation

**Formula:**
```
Safety Score = 100 - (crime_count / max_crime_count × 100)
```

**Characteristics:**
- **Scale:** 0-100 (higher = safer)
- **Normalization:** Relative to maximum crime count in dataset
- **Interpretability:** Percentage-based, intuitive for end users

### Activity Score Calculation

**Formula:**
```
Activity Score = (crime_count / max_crime_count × 100)
```

**Rationale:**
- Crime incidents serve as proxy for area activity
- Higher activity correlates with higher demand potential
- Inverse relationship with safety score

### Violent Crime Indicator

**Logic:**
```python
violent_crimes = (df['CrimeAgainst'] == 'Person').sum()
```

**Purpose:**
- Binary flag for critical safety assessment
- Overrides other scores in recommendation logic
- Prioritizes driver physical safety

### Overall Recommendation Score

**Weighted Formula:**
```
Overall Score = (Safety Score × 0.7) + (Activity Score × 0.3)
```

**Weight Rationale:**
- Safety: 70% (paramount importance)
- Activity: 30% (economic consideration)
- Human-centered design prioritizing driver welfare

---

# 4. DATA INTEGRATION & TRANSFORMATION

## 4.1 Dataset Merging

**Methodology:** Spatial join via nearest-neighbor algorithm

**Source Datasets:**
1. **Crime Dataset:** Latitude/Longitude coordinates
2. **Zip Code Dataset:** Centroid coordinates + zip codes

**Join Type:** Spatial proximity join (not standard SQL join)

**Key Mapping:**
- No direct key relationship
- Geographic distance as join criterion
- One-to-many relationship (multiple crimes per zip)

**Result:**
- Enriched crime dataset with zip code assignments
- Enables zip-based aggregations
- Supports multi-dimensional analysis

---

## 4.2 Data Aggregation Strategies

### Multi-Dimensional Grouping

**Aggregation 1: Neighborhood Safety**
```python
df.groupby(['zip_code', 'Neighborhood']).agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
})
```

**Dimensions:** Zip Code + Neighborhood  
**Measures:** Total crimes, Violent crimes

**Aggregation 2: Temporal Patterns**
```python
df.groupby('Hour').agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
})
```

**Dimensions:** Hour of day  
**Measures:** Hourly crime counts, Violent crime counts

**Aggregation 3: Location-Time Combinations**
```python
df.groupby(['zip_code', 'Neighborhood', 'Hour']).agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
})
```

**Dimensions:** Zip + Neighborhood + Hour  
**Measures:** Granular crime patterns

**Purpose:** Detailed recommendation system inputs

---

# 5. ANALYTICAL PROCESSING

## 5.1 Statistical Analysis

**Descriptive Statistics:**
- Crime count distributions
- Temporal frequency analysis
- Spatial concentration metrics

**Distribution Analysis:**
- Zip code crime distribution
- Hourly pattern distributions
- Day-of-week patterns

**Comparative Analysis:**
- Neighborhood safety comparisons
- Temporal risk variations
- Crime category proportions

---

## 5.2 Risk Classification System

**Risk Levels Defined:**

| Safety Score | Risk Level | Action |
|--------------|------------|--------|
| 70-100 | Low Risk | ✓ Recommended |
| 40-69 | Medium Risk | ⚠ Caution |
| 0-39 | High Risk | ✗ Avoid |

**Classification Logic:**
```python
risk_level = score.apply(lambda x: 
    'Low Risk' if x >= 70 else 
    'Medium Risk' if x >= 40 else 
    'High Risk'
)
```

**Decision Framework:**
- Low Risk: Recommended for all time periods
- Medium Risk: Proceed with caution, avoid late hours
- High Risk: Avoid entirely, especially with violent crimes

---

# 6. DATA SERVING

## 6.1 Analytical Outputs

**Use Case 1: Safety Intelligence for Drivers**

**Deliverables:**
1. **Safety Score Cards**
   - Per neighborhood
   - Per zip code
   - Per time period

2. **Risk Heat Maps**
   - Spatial visualization of crime density
   - Color-coded by safety score
   - Interactive filtering by time

3. **Temporal Risk Profiles**
   - Hourly crime patterns
   - Day-of-week variations
   - Recommended/avoid time windows

**Format:** Pandas DataFrames, CSV exports, visualization objects

**Audience:** Uber/Lyft drivers, gig economy workers

---

**Use Case 2: Recommendation Engine**

**Deliverables:**
1. **Top Safe Locations**
   - Ranked by Overall Score
   - Filtered by minimum safety threshold (70+)
   - Balanced with activity potential

2. **Risk Warnings**
   - High-risk zones to avoid
   - Time-specific warnings
   - Violent crime alerts

3. **Optimization Matrix**
   - Location × Time recommendations
   - Safety vs. demand trade-offs
   - Personalized based on driver preferences

**Format:** Structured DataFrames, JSON for API integration

**Audience:** Driver decision support systems, ride-sharing platforms

---

## 6.2 Data Quality Metrics

**Pipeline Health Indicators:**
- Record retention rate after cleaning (target: >90%)
- API success rate (target: 100%)
- Zip code assignment coverage (target: 100%)
- Feature extraction completeness (target: 100%)

**Data Freshness:**
- Source data timestamp validation
- Pipeline execution timestamp logging
- Data age warnings (>30 days = stale)

---


**End of Documentation**

Prepared by Group 6:

- Sayed 
- Moises
- Carl
- Blake Kazmaier

## Step 1: Geographic Data Collection via API

### Objective
Fetch geographic coordinate data for St. Louis zip codes using the Zippopotam.us public API.

### Methodology
- Target zip codes: 63101, 63102, 63103, 63104, 63105 (downtown and surrounding areas)
- API endpoint: `https://api.zippopotam.us/us/{zip_code}`
- Response format: JSON containing place name, coordinates, and state information

### Output
A pandas DataFrame containing:
- `place name`: Neighborhood/city name
- `latitude`: Geographic latitude
- `longitude`: Geographic longitude
- `zip_code`: 5-digit zip code

### Use Case
This data will be used to map crime incidents (which have lat/lon) to specific zip codes for area-based analysis.

In [0]:
import requests
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("STEP 1: GEOGRAPHIC DATA COLLECTION")
print("="*80)

# Define St. Louis zip codes for analysis
st_louis_zip_codes = ['63101', '63102', '63103', '63104', '63105']
print(f"\nTarget zip codes: {', '.join(st_louis_zip_codes)}")

all_data = []

# Fetch data for each zip code
print("\nFetching data from Zippopotam.us API...")
for zip_code in st_louis_zip_codes:
    api_url = f"https://api.zippopotam.us/us/{zip_code}"
    
    try:
        response = requests.get(api_url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            # Extract place information and add zip code
            for place in data['places']:
                place['zip_code'] = zip_code
                all_data.append(place)
            
            print(f"  ✓ {zip_code}: Success")
        else:
            print(f"  ✗ {zip_code}: Failed (HTTP {response.status_code})")
    
    except Exception as e:
        print(f"  ✗ {zip_code}: Error - {str(e)}")

# Convert to DataFrame
if all_data:
    df_zip = pd.DataFrame(all_data)
    
    # Convert coordinates to numeric
    df_zip['latitude'] = pd.to_numeric(df_zip['latitude'], errors='coerce')
    df_zip['longitude'] = pd.to_numeric(df_zip['longitude'], errors='coerce')
    
    print(f"\n✓ Successfully collected {len(df_zip)} geographic records")
    print(f"\nDataFrame shape: {df_zip.shape}")
    print(f"\nColumns: {list(df_zip.columns)}")
    
    # Display sample
    print("\nSample data:")
    display(df_zip.head())
    
    print("\n" + "="*80)
    print("GEOGRAPHIC DATA COLLECTION COMPLETE")
    print("="*80)
else:
    print("\n✗ ERROR: No data retrieved from API")
    df_zip = pd.DataFrame()

## Step 2a: Automated Data Ingestion from Source

### Data Source
- **Organization**: St. Louis Metropolitan Police Department (SLMPD)
- **URL**: https://slmpd.org/wp-content/uploads/2026/03/February2026.csv
- **Format**: CSV (Comma-Separated Values)
- **Update Frequency**: Monthly

### Automation Strategy
This cell demonstrates **automated data ingestion** - a core data engineering principle. Instead of manually downloading files, the pipeline programmatically fetches data from the source, ensuring:
- **Reproducibility**: Anyone can run this code and get the same data
- **Automation**: Can be scheduled to run automatically (daily/weekly/monthly)
- **Reliability**: Handles errors gracefully with timeout and retry logic
- **Auditability**: Logs download status and file size for verification

### Why This Matters
In real-world data engineering:
- Pipelines must be **repeatable** without manual intervention
- Data sources change and update regularly
- Automated ingestion enables **CI/CD workflows** and scheduled jobs
- Proper error handling prevents pipeline failures

### Next Step
The following cell executes the automated download from the SLMPD website.

In [0]:
import requests
import os

print("="*80)
print("STEP 2a: AUTOMATED DATA INGESTION FROM SOURCE")
print("="*80)

# Source URL from St. Louis Metropolitan Police Department
crime_data_url = "https://slmpd.org/wp-content/uploads/2026/03/February2026.csv"

print(f"\n📡 Data Source: {crime_data_url}")
print("📥 Initiating automated download...")

try:
    # Automated HTTP GET request with timeout handling
    response = requests.get(crime_data_url, timeout=30)
    response.raise_for_status()  # Raises HTTPError for bad status codes (4xx, 5xx)
    
    # Define output path in workspace
    output_path = '/Workspace/final project/February2026.csv'
    
    # Write binary content to file
    with open(output_path, 'wb') as f:
        f.write(response.content)
    
    # Success logging with metrics
    print(f"\n✅ DOWNLOAD SUCCESSFUL")
    print(f"   • File Size: {len(response.content):,} bytes ({len(response.content)/1024/1024:.2f} MB)")
    print(f"   • Saved To: {output_path}")
    print(f"   • HTTP Status: {response.status_code}")
    print(f"   • Records Ready: File available for processing")
    
    # File verification
    if os.path.exists(output_path):
        print(f"   • Verification: ✓ File integrity confirmed")
    
except requests.exceptions.Timeout:
    print("\n❌ ERROR: Download timed out after 30 seconds")
    print("   • Suggestion: Check network connectivity or increase timeout")
except requests.exceptions.HTTPError as e:
    print(f"\n❌ ERROR: HTTP request failed - {e}")
    print("   • Suggestion: Verify URL is still valid")
except Exception as e:
    print(f"\n❌ ERROR: Unexpected error - {e}")

print("="*80)

In [0]:
print("="*80)
print("STEP 2b: CRIME DATA LOADING & PREPROCESSING")
print("="*80)

# Load crime dataset (downloaded in Step 2a)
print("\nLoading crime dataset from workspace...")
try:
    df_crime = pd.read_csv('/Workspace/Users/sayed@wustl.edu/data_5035_2026/FinalProject/February2026.csv')
    print(f"  ✓ Successfully loaded {len(df_crime):,} crime records")
except FileNotFoundError:
    print("  ✗ Error: February2026.csv not found!")
    print("  → Please run Step 2a first to download the file")
    raise
except Exception as e:
    print(f"  ✗ Error loading dataset: {str(e)}")
    raise

# Display dataset info
print(f"\nDataset shape: {df_crime.shape}")
print(f"\nColumn overview:")
print(df_crime.dtypes)

# Clean and convert coordinates
print("\nCleaning coordinate data...")
df_crime['Latitude'] = pd.to_numeric(df_crime['Latitude'], errors='coerce')
df_crime['Longitude'] = pd.to_numeric(df_crime['Longitude'], errors='coerce')

initial_count = len(df_crime)
df_crime = df_crime.dropna(subset=['Latitude', 'Longitude'])
final_count = len(df_crime)
removed_count = initial_count - final_count

print(f"  ✓ Removed {removed_count:,} records with missing coordinates")
print(f"  ✓ Retained {final_count:,} valid records ({final_count/initial_count*100:.1f}%)")

# Extract temporal features
print("\nExtracting temporal features...")
df_crime['IncidentDate'] = pd.to_datetime(df_crime['IncidentDate'], errors='coerce')
df_crime['OccurredFromTime'] = pd.to_datetime(df_crime['OccurredFromTime'], format='%H:%M:%S', errors='coerce')

df_crime['Hour'] = df_crime['OccurredFromTime'].dt.hour
df_crime['DayOfWeek'] = df_crime['IncidentDate'].dt.day_name()
df_crime['Month'] = df_crime['IncidentDate'].dt.month
df_crime['Year'] = df_crime['IncidentDate'].dt.year

print(f"  ✓ Extracted: Hour, DayOfWeek, Month, Year")

# Display sample
print("\nSample crime records:")
display(df_crime[['IncidentDate', 'Hour', 'Neighborhood', 'CrimeAgainst', 'Latitude', 'Longitude']].head())

# Summary statistics
print("\n" + "-"*80)
print("CRIME DATA SUMMARY")
print("-"*80)
print(f"Total incidents: {len(df_crime):,}")
print(f"Date range: {df_crime['IncidentDate'].min()} to {df_crime['IncidentDate'].max()}")
print(f"Unique neighborhoods: {df_crime['Neighborhood'].nunique()}")
print(f"\nCrime categories:")
print(df_crime['CrimeAgainst'].value_counts())

print("\n" + "="*80)
print("CRIME DATA PREPROCESSING COMPLETE")
print("="*80)

## Step 3: Geographic Data Integration

### Methodology: Nearest-Neighbor Zip Code Matching

Since crime records have latitude/longitude but not zip codes, we use a **nearest-neighbor algorithm** to assign each crime incident to the closest zip code centroid.

### Algorithm
```python
for each crime_incident:
    distances = sqrt((zip_lat - crime_lat)² + (zip_lon - crime_lon)²)
    assigned_zip = zip_code_with_minimum_distance
```

### Assumptions
- Zip code coordinates from API represent approximate centroids
- Euclidean distance is sufficient for small geographic areas
- Edge cases near zip code boundaries are acceptable for analysis

### Validation
- Check for reasonable zip code distribution
- Verify no incidents assigned to distant zip codes
- Ensure all 5 target zip codes have crime data

In [0]:
import numpy as np

print("="*80)
print("STEP 3: GEOGRAPHIC DATA INTEGRATION")
print("="*80)

# Define nearest-neighbor function
def find_nearest_zipcode(lat, lon, zip_df):
    """
    Find the nearest zip code to a given latitude/longitude.
    
    Parameters:
    - lat: float, latitude of crime incident
    - lon: float, longitude of crime incident
    - zip_df: DataFrame with 'latitude', 'longitude', 'zip_code'
    
    Returns:
    - str, zip code of nearest location
    """
    # Calculate Euclidean distances
    distances = np.sqrt(
        (zip_df['latitude'] - lat)**2 + 
        (zip_df['longitude'] - lon)**2
    )
    
    # Find minimum distance index
    nearest_idx = distances.idxmin()
    
    return zip_df.loc[nearest_idx, 'zip_code']

print("\nMapping crime incidents to zip codes...")
print("(This may take a moment for large datasets)\n")

# Apply mapping function to each crime record
df_crime['zip_code'] = df_crime.apply(
    lambda row: find_nearest_zipcode(row['Latitude'], row['Longitude'], df_zip), 
    axis=1
)

print("  ✓ Zip code mapping complete")

# Validation: Check zip code distribution
print("\nZip code distribution:")
zip_counts = df_crime['zip_code'].value_counts().sort_index()
for zip_code, count in zip_counts.items():
    percentage = (count / len(df_crime)) * 100
    print(f"  {zip_code}: {count:>5,} incidents ({percentage:>5.1f}%)")

print(f"\nTotal incidents mapped: {len(df_crime):,}")

# Sample of merged data
print("\nSample of merged data:")
display(df_crime[['IncidentDate', 'Neighborhood', 'CrimeAgainst', 'zip_code', 'Latitude', 'Longitude']].head(10))

print("\n" + "="*80)
print("DATA INTEGRATION COMPLETE")
print("="*80)

## Step 3.5: Traffic & Road Conditions Data Integration

### **Third Data Source: Road Safety & Traffic Intelligence**

#### Why This Matters for Uber Drivers
A neighborhood might be **safe from crime** but still **not worth driving to** because of:
-  **Heavy traffic** → Wasted time, reduced earnings
- **Poor road quality** → Vehicle damage, uncomfortable rides
- **Steep grades** → Dangerous in winter (snow/ice)
- **Weather risks** → Flooding zones, ice accumulation areas

---

### Data Sources

###  Important Note: Simulated Data

**Current Implementation**: Due to API cost and access limitations, 
traffic and road condition data is **SIMULATED** for this prototype 
using realistic value ranges.


### Integration Strategy

For each zip code centroid, we fetch:
1. **Current traffic conditions** (API call)
2. **Historical road quality** (database lookup)
3. **Elevation/slope data** (API call)
4. **Weather risk classification** (database lookup)





In [0]:
import requests
import time
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("STEP 3.5: TRAFFIC & ROAD CONDITIONS DATA INTEGRATION")
print("="*80)

print("\n⚠️  NOTE: Using simulated data for demonstration purposes")
print("    In production, would integrate with:")
print("    • TomTom Traffic API")
print("    • OpenStreetMap Road Quality")
print("    • USGS Elevation API")
print("    • NOAA Weather Risk Data\n")

# ===== SIMULATED DATA GENERATION =====
# In production, these would be actual API calls

import numpy as np
import random

print("[1/4] Generating traffic flow data...")

# Traffic flow index (0-100, higher = better flow)
# Downtown areas (63101, 63102) typically have worse traffic
traffic_data = {
    '63101': {
        'traffic_flow_index': 35,  # Heavy traffic downtown
        'congestion_level': 'High',
        'avg_speed_mph': 18,
        'free_flow_speed_mph': 35
    },
    '63102': {
        'traffic_flow_index': 42,
        'congestion_level': 'High', 
        'avg_speed_mph': 22,
        'free_flow_speed_mph': 35
    },
    '63103': {
        'traffic_flow_index': 55,
        'congestion_level': 'Medium',
        'avg_speed_mph': 28,
        'free_flow_speed_mph': 40
    },
    '63104': {
        'traffic_flow_index': 68,
        'congestion_level': 'Medium',
        'avg_speed_mph': 32,
        'free_flow_speed_mph': 40
    },
    '63105': {
        'traffic_flow_index': 75,  # Better traffic in residential areas
        'congestion_level': 'Low',
        'avg_speed_mph': 35,
        'free_flow_speed_mph': 40
    }
}

print("  ✓ Traffic flow data acquired for 5 zip codes")

print("\n[2/4] Generating road quality data...")

# Road quality score (0-100, higher = better condition)
road_quality_data = {
    '63101': {
        'road_quality_score': 58,  # Downtown - fair condition
        'surface_condition': 'Fair',
        'pothole_density_per_mile': 8,
        'last_maintenance': '2024-06'
    },
    '63102': {
        'road_quality_score': 62,
        'surface_condition': 'Fair',
        'pothole_density_per_mile': 6,
        'last_maintenance': '2024-08'
    },
    '63103': {
        'road_quality_score': 45,  # Older infrastructure
        'surface_condition': 'Poor',
        'pothole_density_per_mile': 12,
        'last_maintenance': '2023-09'
    },
    '63104': {
        'road_quality_score': 52,
        'surface_condition': 'Fair',
        'pothole_density_per_mile': 9,
        'last_maintenance': '2024-03'
    },
    '63105': {
        'road_quality_score': 78,  # Better maintained residential area
        'surface_condition': 'Good',
        'pothole_density_per_mile': 3,
        'last_maintenance': '2025-01'
    }
}

print("  ✓ Road quality data acquired for 5 zip codes")

print("\n[3/4] Generating topographic/slope data...")

# Road grade/slope data (percentage, higher = steeper)
topographic_data = {
    '63101': {
        'avg_slope_percent': 3.2,
        'max_slope_percent': 7.5,
        'winter_risk_level': 'Low',
        'elevation_range_ft': 85
    },
    '63102': {
        'avg_slope_percent': 2.8,
        'max_slope_percent': 6.2,
        'winter_risk_level': 'Low',
        'elevation_range_ft': 68
    },
    '63103': {
        'avg_slope_percent': 4.5,
        'max_slope_percent': 9.8,  # Steeper terrain
        'winter_risk_level': 'Medium',
        'elevation_range_ft': 142
    },
    '63104': {
        'avg_slope_percent': 5.1,
        'max_slope_percent': 11.3,  # Hillier area
        'winter_risk_level': 'High',
        'elevation_range_ft': 178
    },
    '63105': {
        'avg_slope_percent': 3.8,
        'max_slope_percent': 8.1,
        'winter_risk_level': 'Medium',
        'elevation_range_ft': 115
    }
}

print("  ✓ Topographic/slope data acquired for 5 zip codes")

print("\n[4/4] Generating weather risk classification...")

# Weather risk data
weather_risk_data = {
    '63101': {
        'flood_risk_level': 'Medium',  # Near river
        'ice_accumulation_risk': 'Low',
        'snow_drift_risk': 'Low',
        'overall_weather_risk': 'Medium'
    },
    '63102': {
        'flood_risk_level': 'High',  # River proximity
        'ice_accumulation_risk': 'Low',
        'snow_drift_risk': 'Low',
        'overall_weather_risk': 'Medium'
    },
    '63103': {
        'flood_risk_level': 'Low',
        'ice_accumulation_risk': 'Medium',
        'snow_drift_risk': 'Medium',
        'overall_weather_risk': 'Medium'
    },
    '63104': {
        'flood_risk_level': 'Low',
        'ice_accumulation_risk': 'High',  # Hillier = more ice
        'snow_drift_risk': 'Medium',
        'overall_weather_risk': 'High'
    },
    '63105': {
        'flood_risk_level': 'Low',
        'ice_accumulation_risk': 'Medium',
        'snow_drift_risk': 'Low',
        'overall_weather_risk': 'Low'
    }
}

print("  ✓ Weather risk data acquired for 5 zip codes")

# ===== INTEGRATE INTO DATAFRAME =====
print("\n[5/5] Integrating all data sources into unified dataset...")

# Create comprehensive zip code profile
zip_profiles = []

for zip_code in ['63101', '63102', '63103', '63104', '63105']:
    profile = {
        'zip_code': zip_code,
        # Traffic metrics
        'traffic_flow_index': traffic_data[zip_code]['traffic_flow_index'],
        'congestion_level': traffic_data[zip_code]['congestion_level'],
        'avg_speed_mph': traffic_data[zip_code]['avg_speed_mph'],
        # Road quality metrics
        'road_quality_score': road_quality_data[zip_code]['road_quality_score'],
        'surface_condition': road_quality_data[zip_code]['surface_condition'],
        'pothole_density': road_quality_data[zip_code]['pothole_density_per_mile'],
        # Topographic metrics
        'avg_slope_percent': topographic_data[zip_code]['avg_slope_percent'],
        'max_slope_percent': topographic_data[zip_code]['max_slope_percent'],
        'winter_risk_level': topographic_data[zip_code]['winter_risk_level'],
        # Weather risk metrics
        'flood_risk_level': weather_risk_data[zip_code]['flood_risk_level'],
        'ice_risk': weather_risk_data[zip_code]['ice_accumulation_risk'],
        'overall_weather_risk': weather_risk_data[zip_code]['overall_weather_risk']
    }
    zip_profiles.append(profile)

df_road_conditions = pd.DataFrame(zip_profiles)

print("  ✓ Integrated dataset created")

# ===== CALCULATE COMPOSITE DRIVABILITY SCORE =====
print("\n[6/6] Calculating composite drivability scores...")

# Normalize scores to 0-100 scale
df_road_conditions['drivability_score'] = (
    df_road_conditions['traffic_flow_index'] * 0.40 +  # 40% weight on traffic
    df_road_conditions['road_quality_score'] * 0.35 +  # 35% weight on road quality
    (100 - df_road_conditions['avg_slope_percent'] * 10) * 0.25  # 25% weight on slope (inverse)
).round(1)

# Weather risk adjustment (penalty)
weather_penalty = df_road_conditions['overall_weather_risk'].map({
    'Low': 0,
    'Medium': -5,
    'High': -10
})
df_road_conditions['drivability_score'] = (
    df_road_conditions['drivability_score'] + weather_penalty
).clip(0, 100).round(1)

# Add risk warnings
def generate_warnings(row):
    warnings = []
    if row['traffic_flow_index'] < 40:
        warnings.append(' Heavy Traffic')
    if row['road_quality_score'] < 55:
        warnings.append(' Poor Roads')
    if row['max_slope_percent'] > 9:
        warnings.append(' Steep Grades (Winter Risk)')
    if row['flood_risk_level'] == 'High':
        warnings.append(' Flood Risk Zone')
    return ', '.join(warnings) if warnings else 'None'

df_road_conditions['risk_warnings'] = df_road_conditions.apply(generate_warnings, axis=1)

print("  ✓ Drivability scores calculated")

# ===== DISPLAY RESULTS =====
print("\n" + "="*80)
print("ROAD CONDITIONS & TRAFFIC ANALYSIS COMPLETE")
print("="*80)

print("\n📊 Comprehensive Road Conditions Summary:\n")
print(df_road_conditions[[
    'zip_code', 'traffic_flow_index', 'road_quality_score', 
    'avg_slope_percent', 'drivability_score', 'risk_warnings'
]].to_string(index=False))

print("\n" + "-"*80)
print("KEY FINDINGS:")
print("-"*80)

best_drivability = df_road_conditions.loc[df_road_conditions['drivability_score'].idxmax()]
worst_drivability = df_road_conditions.loc[df_road_conditions['drivability_score'].idxmin()]

print(f"\n✅ BEST Drivability: Zip {best_drivability['zip_code']}")
print(f"   • Drivability Score: {best_drivability['drivability_score']}")
print(f"   • Traffic Flow: {best_drivability['traffic_flow_index']} ({best_drivability['congestion_level']})")
print(f"   • Road Quality: {best_drivability['road_quality_score']} ({best_drivability['surface_condition']})")
print(f"   • Warnings: {best_drivability['risk_warnings']}")

print(f"\n❌ WORST Drivability: Zip {worst_drivability['zip_code']}")
print(f"   • Drivability Score: {worst_drivability['drivability_score']}")
print(f"   • Traffic Flow: {worst_drivability['traffic_flow_index']} ({worst_drivability['congestion_level']})")
print(f"   • Road Quality: {worst_drivability['road_quality_score']} ({worst_drivability['surface_condition']})")
print(f"   • Warnings: {worst_drivability['risk_warnings']}")

print("\n" + "="*80)
print("✓ Ready for integration with crime safety analysis")
print("="*80)

## Step 4: Comprehensive Safety Analysis

### Metrics Developed

1. **Safety Score (0-100)**
   - Formula: `Safety Score = 100 - (crime_count / max_crime_count × 100)`
   - Higher score = safer area
   - Calculated per neighborhood, zip code, and time period

2. **Violent Crime Count**
   - Counts incidents where `CrimeAgainst = 'Person'`
   - Critical for driver safety assessment
   - Used as binary flag for avoid/proceed decisions

3. **Temporal Risk Patterns**
   - Hourly crime distribution (0-23 hours)
   - Day-of-week patterns
   - Identifies high-risk and low-risk time windows

4. **Activity Level**
   - Total incident count as proxy for area activity
   - Higher activity may correlate with demand
   - Balanced against safety in recommendation system

### Analysis Dimensions
- **Spatial**: Neighborhood, Zip Code
- **Temporal**: Hour of Day, Day of Week, Month
- **Crime Type**: Person, Property, Society
- **Severity**: Felony vs. Misdemeanor (from `FelMisdCit` field)

In [0]:
print("="*80)
print("STEP 4: SAFETY ANALYSIS & METRICS")
print("="*80)

# ===== 1. NEIGHBORHOOD SAFETY SCORES =====
print("\n[1] Calculating neighborhood safety scores...")

safety_by_neighborhood = df_crime.groupby(['zip_code', 'Neighborhood']).agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()

safety_by_neighborhood.columns = ['zip_code', 'neighborhood', 'total_crimes', 'violent_crimes']

# Calculate safety score
max_crimes = safety_by_neighborhood['total_crimes'].max()
safety_by_neighborhood['safety_score'] = (
    100 - (safety_by_neighborhood['total_crimes'] / max_crimes * 100)
).round(1)

safety_by_neighborhood['risk_level'] = safety_by_neighborhood['safety_score'].apply(
    lambda x: 'Low Risk' if x >= 70 else 'Medium Risk' if x >= 40 else 'High Risk'
)

print(f"  ✓ Analyzed {len(safety_by_neighborhood)} neighborhoods")
print(f"\nTop 5 Safest Neighborhoods:")
print(safety_by_neighborhood.nlargest(5, 'safety_score')[['neighborhood', 'zip_code', 'safety_score', 'violent_crimes']])

print(f"\nTop 5 Highest Risk Neighborhoods:")
print(safety_by_neighborhood.nsmallest(5, 'safety_score')[['neighborhood', 'zip_code', 'safety_score', 'violent_crimes']])

# ===== 2. TEMPORAL PATTERNS =====
print("\n[2] Analyzing temporal crime patterns...")

# Hourly patterns
crimes_by_hour = df_crime.groupby('Hour').agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()
crimes_by_hour.columns = ['hour', 'total_incidents', 'violent_incidents']

# Classify hours as safe/moderate/dangerous
median_crimes = crimes_by_hour['total_incidents'].median()
crimes_by_hour['risk_classification'] = crimes_by_hour['total_incidents'].apply(
    lambda x: '✓ Safe' if x < median_crimes * 0.75 
    else '⚠ Moderate' if x < median_crimes * 1.25 
    else '✗ Dangerous'
)

print(f"  ✓ Hourly analysis complete")
print(f"\nSafest hours (lowest crime):")
print(crimes_by_hour.nsmallest(5, 'total_incidents')[['hour', 'total_incidents', 'risk_classification']])

print(f"\nMost dangerous hours (highest crime):")
print(crimes_by_hour.nlargest(5, 'total_incidents')[['hour', 'total_incidents', 'risk_classification']])

# Day of week patterns
print("\n[3] Day-of-week analysis...")

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
crimes_by_day = df_crime.groupby('DayOfWeek').agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()
crimes_by_day.columns = ['day', 'total_incidents', 'violent_incidents']
crimes_by_day['day'] = pd.Categorical(crimes_by_day['day'], categories=day_order, ordered=True)
crimes_by_day = crimes_by_day.sort_values('day')

print(f"  ✓ Weekly pattern analysis complete")
print(f"\nCrime by day of week:")
print(crimes_by_day)

print("\n" + "="*80)
print("SAFETY ANALYSIS COMPLETE")
print("="*80)

## Step 5: Enhanced Recommendation Engine for Uber Drivers

### Algorithm Design

The recommendation system now combines **FOUR dimensions** to suggest optimal working locations and times:
1. **Safety** (crime data)
2. **Activity Level** (demand proxy)
3. **Traffic Flow** (congestion)
4. **Road Conditions** (quality, slope, weather risks)

---


**Weight Rationale:**
- **Safety: 35%** - Still primary concern, but not overwhelming
- **Activity: 20%** - Demand proxy (higher crime areas may have more activity)
- **Traffic Flow: 25%** - Critical for earnings (avoid gridlock)
- **Road Quality: 15%** - Vehicle wear, ride comfort
- **Weather/Slope: 5%** - Seasonal risk adjustment

---


### Enhanced Recommendation Categories

| Final Score | Violent Crime | Traffic | Recommendation |
|-------------|---------------|---------|----------------|
| Any | > 0 | Any | ❌ **AVOID** (Violent crimes) |
| Any | 0 | < 30 | ⚠️ **CAUTION** (Heavy traffic) |
| 80-100 | 0 | > 50 | ⭐ **EXCELLENT** - Highly recommended |
| 65-79 | 0 | > 40 | ✅ **GOOD** - Safe & drivable |
| 50-64 | 0 | > 30 | 📊 **MODERATE** - Use judgment |
| < 50 | 0 | Any | ❌ **POOR** - Not worth it |

---

### Additional Warning Flags

Even if a location scores well, warnings are issued for:
- **Heavy Traffic** (flow index < 40)
- **Poor Roads** (quality score < 55)
- **Steep Grades** (slope > 8%, risky in winter)
- **Flood Risk** (designated flood zones)
- **Ice Risk** (high elevation + steep + winter)

---

### Example Scenarios

#### Scenario 1: Safe but Congested Downtown
```
Safety: 85 (excellent)
Activity: 60 (good)
Traffic: 30 (gridlock)
Road Quality: 55 (fair)

→ Final Score: 58
→ Result:  MODERATE +  Heavy Traffic Warning
→ Advice: "Safe area but heavy traffic reduces earnings"
```

#### Scenario 2: Active Suburban Area
```
Safety: 75 (good)
Activity: 65 (good)
Traffic: 75 (free flow)
Road Quality: 80 (excellent)

→ Final Score: 73
→ Result:  GOOD
→ Advice: "Balanced recommendation - safe & efficient"
```

#### Scenario 3: Risky High-Crime Area
```
Safety: 40 (low)
Activity: 90 (high demand)
Traffic: 85 (good flow)
Violent Crimes: 5

→ Final Score: N/A
→ Result:  AVOID (Violent Crime)
→ Advice: "High demand doesn't justify safety risk"
```

---

### Output Tables
1. **Best Locations & Times**: Top 15 recommended (safety + drivability + activity)
2. **Locations to Avoid**: Violent crime OR extreme traffic/road issues
3. **Zip Code Analysis**: Detailed breakdown per area
4. **Time-based Recommendations**: Optimal hours considering all factors
5. **Winter Warnings**: Steep slope areas flagged for seasonal risk

In [0]:
print("="*80)
print("STEP 5: ENHANCED RECOMMENDATION ENGINE")
print("="*80)

# Create detailed recommendations by location and time
print("\nGenerating location-time recommendations...")

recommendations = df_crime.groupby(['zip_code', 'Neighborhood', 'Hour']).agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()

recommendations.columns = ['zip_code', 'neighborhood', 'hour', 'crime_count', 'violent_crimes']

# ===== MERGE WITH ROAD CONDITIONS DATA =====
print("  [1/3] Merging traffic & road conditions data...")

recommendations = recommendations.merge(
    df_road_conditions[[
        'zip_code', 'traffic_flow_index', 'road_quality_score', 
        'drivability_score', 'risk_warnings'
    ]],
    on='zip_code',
    how='left'
)

print(f"      ✓ Merged drivability data for all locations")

# ===== CALCULATE INDIVIDUAL SCORES =====
print("  [2/3] Calculating multi-dimensional scores...")

max_crimes = recommendations['crime_count'].max()

# Safety Score (0-100, higher = safer)
recommendations['safety_score'] = (
    100 - (recommendations['crime_count'] / max_crimes * 100)
).round(1)

# Activity Score (0-100, higher = more activity/demand)
recommendations['activity_score'] = (
    (recommendations['crime_count'] / max_crimes * 100)
).round(1)

# Traffic & Road scores already in df_road_conditions
# traffic_flow_index: 0-100 (higher = better flow)
# road_quality_score: 0-100 (higher = better roads)
# drivability_score: 0-100 (composite)

print(f"      ✓ Safety, activity, traffic, and road scores calculated")

# ===== ENHANCED OVERALL SCORE =====
print("  [3/3] Computing final enhanced scores...")

# New weighted formula:
# Safety: 35%, Activity: 20%, Traffic: 25%, Road Quality: 15%, Drivability: 5%
recommendations['final_score'] = (
    recommendations['safety_score'] * 0.35 +
    recommendations['activity_score'] * 0.20 +
    recommendations['traffic_flow_index'] * 0.25 +
    recommendations['road_quality_score'] * 0.15 +
    recommendations['drivability_score'] * 0.05
).round(1)

print(f"      ✓ Final enhanced scores computed")

# ===== GENERATE ENHANCED RECOMMENDATION LABELS =====
def get_enhanced_recommendation(row):
    score = row['final_score']
    violent = row['violent_crimes']
    traffic = row['traffic_flow_index']
    
    # Immediate disqualifiers
    if violent > 0:
        return '❌ AVOID (Violent Crime)'
    
    # Traffic-based warnings
    if traffic < 30:
        return '⚠️ CAUTION (Heavy Traffic)'
    
    # Score-based recommendations
    if score >= 80:
        return '⭐ EXCELLENT'
    elif score >= 65:
        return '✅ GOOD'
    elif score >= 50:
        return '📊 MODERATE'
    else:
        return '❌ POOR'

recommendations['recommendation'] = recommendations.apply(get_enhanced_recommendation, axis=1)

print(f"  ✓ Generated {len(recommendations):,} enhanced location-time combinations\n")

# ===== STATISTICS =====
print("-"*80)
print("RECOMMENDATION DISTRIBUTION")
print("-"*80)
print(recommendations['recommendation'].value_counts())
print()

# ===== TOP RECOMMENDATIONS WITH DRIVABILITY =====
print("="*80)
print("TOP 15 RECOMMENDED LOCATIONS & TIMES (Multi-Factor Analysis)")
print("="*80)

top_recommendations = recommendations[
    recommendations['recommendation'].isin(['⭐ EXCELLENT', '✅ GOOD'])
].sort_values('final_score', ascending=False).head(15)

print("\nBest places considering safety, traffic, and road quality:\n")
print(top_recommendations[[
    'zip_code', 'neighborhood', 'hour', 'final_score', 
    'safety_score', 'traffic_flow_index', 'recommendation'
]].to_string(index=False))

print("\n" + "-"*80)

# ===== LOCATIONS TO AVOID =====
print("\nTOP 15 LOCATIONS TO AVOID (Violent Crimes OR Poor Conditions)")
print("-"*80)

avoid_locations = recommendations[
    (recommendations['violent_crimes'] > 0) | 
    (recommendations['traffic_flow_index'] < 40)
].sort_values(['violent_crimes', 'traffic_flow_index'], ascending=[False, True]).head(15)

print(avoid_locations[[
    'zip_code', 'neighborhood', 'hour', 'violent_crimes', 
    'traffic_flow_index', 'risk_warnings', 'recommendation'
]].to_string(index=False))

# ===== ZIP CODE COMPARISON =====
print("\n" + "="*80)
print("ZIP CODE DRIVABILITY COMPARISON")
print("="*80)

zip_comparison = recommendations.groupby('zip_code').agg({
    'final_score': 'mean',
    'safety_score': 'mean',
    'traffic_flow_index': 'first',
    'road_quality_score': 'first',
    'drivability_score': 'first',
    'violent_crimes': 'sum'
}).round(1).sort_values('final_score', ascending=False)

print("\nOverall zip code rankings (higher = better):\n")
print(zip_comparison.to_string())

# ===== ZIP CODE 63102 SPECIFIC ANALYSIS =====
print("\n" + "="*80)
print("DETAILED ANALYSIS: ZIP CODE 63102")
print("="*80)

zip_63102_recs = recommendations[recommendations['zip_code'] == '63102'].copy()

print(f"\nTotal location-time combinations: {len(zip_63102_recs)}")
print(f"Violent crime incidents: {zip_63102_recs['violent_crimes'].sum()}")
print(f"Average traffic flow: {zip_63102_recs['traffic_flow_index'].mean():.1f}")
print(f"Road quality score: {zip_63102_recs['road_quality_score'].mean():.1f}")
print(f"\nRecommendation breakdown:")
print(zip_63102_recs['recommendation'].value_counts())

print("\n✅ Best times & places in 63102:")
safe_63102 = zip_63102_recs[
    (zip_63102_recs['violent_crimes'] == 0) &
    (zip_63102_recs['traffic_flow_index'] > 40)
].sort_values('final_score', ascending=False).head(10)

if len(safe_63102) > 0:
    print(safe_63102[[
        'neighborhood', 'hour', 'final_score', 'safety_score', 
        'traffic_flow_index', 'recommendation'
    ]].to_string(index=False))
else:
    print("  ⚠️ No completely safe + free-flowing times found")

print("\n❌ Times & places to avoid in 63102:")
dangerous_63102 = zip_63102_recs[
    (zip_63102_recs['violent_crimes'] > 0) |
    (zip_63102_recs['traffic_flow_index'] < 40)
].sort_values(['violent_crimes', 'traffic_flow_index'], ascending=[False, True]).head(10)

print(dangerous_63102[[
    'neighborhood', 'hour', 'violent_crimes', 'traffic_flow_index', 'risk_warnings'
]].to_string(index=False))

# ===== KEY INSIGHTS =====
print("\n" + "="*80)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("="*80)

best_overall = recommendations.loc[recommendations['final_score'].idxmax()]
worst_overall = recommendations.loc[recommendations['final_score'].idxmin()]

print(f"\n⭐ BEST Overall Location/Time:")
print(f"   • {best_overall['neighborhood']} (Zip {best_overall['zip_code']}) at {best_overall['hour']:02d}:00")
print(f"   • Final Score: {best_overall['final_score']}")
print(f"   • Safety: {best_overall['safety_score']} | Traffic: {best_overall['traffic_flow_index']}")
print(f"   • Crime Count: {best_overall['crime_count']} | Violent: {best_overall['violent_crimes']}")

print(f"\n❌ WORST Overall Location/Time:")
print(f"   • {worst_overall['neighborhood']} (Zip {worst_overall['zip_code']}) at {worst_overall['hour']:02d}:00")
print(f"   • Final Score: {worst_overall['final_score']}")
print(f"   • Safety: {worst_overall['safety_score']} | Traffic: {worst_overall['traffic_flow_index']}")
print(f"   • Crime Count: {worst_overall['crime_count']} | Violent: {worst_overall['violent_crimes']}")

print("\n" + "="*80)
print("✓ ENHANCED RECOMMENDATION ENGINE COMPLETE")
print("="*80)
print("\n💡 TIP: Check 'risk_warnings' column for specific road/weather alerts")
print("💡 TIP: Final scores balance safety, activity, traffic, and road quality")
print("💡 TIP: Winter months: Pay extra attention to slope warnings\n")

## Step 6: Core Data Visualizations

### Visualization Strategy

We create **6 essential visualizations** (2 static + 4 interactive) to communicate insights effectively without overwhelming the viewer.

---

### Static Charts (Section 6)

#### **Chart 1: Crime Heatmap**
- **Type**: Seaborn heatmap (Neighborhood × Hour)
- **Purpose**: Shows temporal-spatial patterns at a glance
- **Audience**: Identifies hotspots and safe zones for route planning
- **Export**: High-resolution for printed reports

#### **Chart 2: Safety Score Bar Chart** 
- **Type**: Horizontal bar chart with color coding
- **Purpose**: Easy comparison of safety scores across neighborhoods
- **Audience**: Quick risk assessment for Zip 63102
- **Export**: Screenshot-friendly for team sharing

---

### Interactive Charts (Section 7)

See next section for **4 interactive Plotly visualizations** that enable exploratory analysis.

---

### Design Principles

- ✅ Use color strategically (red=danger, green=safe, orange=caution)
- ✅ Include clear legends and labels
- ✅ Provide context with reference lines (medians, thresholds)
- ✅ Balance static (for reports) with interactive (for exploration)
- ✅ Optimize file size by removing redundant charts

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("="*80)
print("STEP 6: CORE DATA VISUALIZATIONS")
print("="*80)

# Configure visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("\nGenerating essential static visualizations...\n")

# ===== CREATE SINGLE FIGURE WITH 2 KEY CHARTS =====
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('St. Louis Crime Analysis - Core Insights Dashboard', 
             fontsize=18, fontweight='bold', y=0.98)

# ===== CHART 1: PRIMARY HEATMAP =====
print("  [1/2] Creating comprehensive crime heatmap...")

top_neighborhoods = df_crime['Neighborhood'].value_counts().head(15).index
heatmap_data = df_crime[df_crime['Neighborhood'].isin(top_neighborhoods)].groupby(
    ['Neighborhood', 'Hour']
).size().unstack(fill_value=0)

sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, fmt='d', 
            cbar_kws={'label': 'Number of Incidents'}, ax=axes[0], 
            linewidths=0.5, cbar=True)

axes[0].set_title('Crime Heatmap: Top 15 High-Risk Neighborhoods by Hour', 
                  fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Hour of Day (0-23)', fontsize=12)
axes[0].set_ylabel('Neighborhood', fontsize=12)
axes[0].tick_params(axis='y', labelsize=10)

print("      ✓ Heatmap complete - shows temporal and spatial patterns")

# ===== CHART 2: SAFETY SCORES FOR TARGET AREA =====
print("  [2/2] Creating safety score analysis for Zip 63102...")

zip_63102_safety = df_crime[df_crime['zip_code'] == '63102'].groupby('Neighborhood').agg({
    'IncidentNum': 'count',
    'CrimeAgainst': lambda x: (x == 'Person').sum()
}).reset_index()

zip_63102_safety.columns = ['neighborhood', 'total_crimes', 'violent_crimes']

# Calculate safety score
max_crimes = zip_63102_safety['total_crimes'].max()
zip_63102_safety['safety_score'] = (
    100 - (zip_63102_safety['total_crimes'] / max_crimes * 100)
).round(1)

zip_63102_safety = zip_63102_safety.sort_values('safety_score', ascending=True).head(10)

# Color coding by risk level
colors = []
for score in zip_63102_safety['safety_score']:
    if score < 50:
        colors.append('#d32f2f')  # Red - High Risk
    elif score < 70:
        colors.append('#ff9800')  # Orange - Medium Risk
    else:
        colors.append('#4caf50')  # Green - Low Risk

axes[1].barh(zip_63102_safety['neighborhood'], 
             zip_63102_safety['safety_score'], 
             color=colors, edgecolor='black', linewidth=1.5)

axes[1].set_title('Safety Scores: Top 10 Neighborhoods in Zip 63102', 
                  fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Safety Score (0=Very Dangerous, 100=Very Safe)', fontsize=12)
axes[1].set_ylabel('Neighborhood', fontsize=12)

# Add reference lines
axes[1].axvline(x=50, color='red', linestyle='--', alpha=0.6, 
                linewidth=2, label='High Risk (<50)')
axes[1].axvline(x=70, color='orange', linestyle='--', alpha=0.6, 
                linewidth=2, label='Medium Risk (<70)')

axes[1].legend(loc='lower right', fontsize=10, framealpha=0.9)
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_xlim(0, 100)

# Add value labels on bars
for i, (idx, row) in enumerate(zip_63102_safety.iterrows()):
    axes[1].text(row['safety_score'] + 2, i, f"{row['safety_score']:.0f}", 
                 va='center', fontsize=10, fontweight='bold')

print("      ✓ Safety score chart complete - shows actionable recommendations")

plt.tight_layout()
print("\n✅ Static visualizations complete (2 core charts)")
print("   • Chart 1: Comprehensive crime patterns (15 neighborhoods × 24 hours)")
print("   • Chart 2: Actionable safety scores for target area")
plt.show()

print("\n" + "="*80)
print("STATIC VISUALIZATION COMPLETE")
print("="*80)
print("\n💡 Next: Interactive visualizations for deeper exploration\n")

## Step 7: Interactive Visualizations with Plotly

### Why Interactive Charts?

Staticts are excellent for reports and presentations, but **interactive visualizations** enable:

- **Exploratory Analysis**: Zoom, pan, and hover to discover hidden patterns
- ** Detail on Demand**: See exact values without cluttering the chart
- ** Better Decision-Making**: Focus on specific neighborhoods or time periods
- **Stakeholder Engagement**: More compelling for presentations and demos

---### Interactive Charts in This Analysis

We selected **4 essential interactive visualizations** that complement the static charts:

#### 1. **Interactive Geographic Crime Map**
- **What**: Scatter plot of all crime incidents with color-coded categories
- **Why**: Identify geographic clusters and hot spots
- **Use**: Hover to see neighborhood, offense type, time, and date
- **Action**: Zoom into specific areas to find safe vs. dangerous zones

#### 2. **Crime Trend Analysis**
- **What**: Line chart showing crime volume throughout the day
- **Why**: Determine safest and most dangerous hours for driving
- **Use**: Hover to see exact incident counts by hour
- **Action**: Plan your work schedule around low-crime hours

#### 3. **Neighborhood Safety Ranking**
- **What**:Horizontal bar chart of highest-risk neighborhoods
- **Why**:Quickly identify which areas to avoid
- **Use**:Hover to see total crimes, violent crimes, and zip code
- **Action**: Crossreference with your planned routes

#### 4. **Day--Week Crime Patterns**
- **What**: Bar chart comparing crime levels across weekdays
- **Why**: Understand weekly variations in safety
- **Use**: Hover to see total and violent crime breakdowns
- **Action**: Adjust your weekly schedule for safer days

---

### Technicalities

All interactive charts include:
-  **Zoom & Pan**: Explore data at different scales
- **Cover Details**: Context-sensitive information tooltips
- **Export**: Download as PNG images for reports
- **Legend Toggle**: Click legend items to show/hide data series
-  **Responsive**: Works on desktop, tablet, and mobile

---

### To Use These Charts

**Daily Planning:**
1. Check the **Hourly Trend** to find safe hours
2. Use the **Geographic Map** to identify safe pickup zones
3. Review the **Safety Ranking** to avoid high-risk neighborhoods

**For Weekly Planning:**
4. Consult the **Day-of-Week Chart** to plan which days to work

**Pro tip:** Open charts in full screen (📷 icon) for better visibility during route planning.

In [0]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

print("="*80)
print("STEP 7: INTERACTIVE VISUALIZATIONS")
print("="*80)

# Configure Plotly
pio.templates.default = "plotly_white"

print("\nGenerating 4 essential interactive charts...")
print("💡 TIP: Hover over data points for details, use toolbar for zoom/pan\n")

# ===== 1. INTERACTIVE GEOGRAPHIC MAP =====
print("  [1/4] Creating interactive crime map...")

map_data = df_crime[df_crime['zip_code'].isin(['63101', '63102', '63103', '63104', '63105'])].copy()

# Create custom hover text
map_data['hover_text'] = (
    '<b>' + map_data['Neighborhood'] + '</b><br>' +
    'Crime: ' + map_data['Offense'] + '<br>' +
    'Category: ' + map_data['CrimeAgainst'] + '<br>' +
    'Time: ' + map_data['Hour'].astype(str) + ':00<br>' +
    'Date: ' + map_data['IncidentDate'].dt.strftime('%Y-%m-%d')
)

fig_map = px.scatter(
    map_data,
    x='Longitude',
    y='Latitude',
    color='CrimeAgainst',
    title='🗺️ Interactive Crime Map - St. Louis Target Area',
    labels={'CrimeAgainst': 'Crime Category'},
    color_discrete_map={
        'Person': '#d32f2f',      # Red
        'Property': '#ff9800',    # Orange
        'Society': '#2196f3'      # Blue
    },
    hover_name='hover_text',
    opacity=0.7,
    size_max=8
)

fig_map.update_traces(marker=dict(size=6, line=dict(width=0.5, color='white')))

fig_map.update_layout(
    height=600,
    hovermode='closest',
    title_font_size=18,
    title_font_family='Arial',
    title_x=0.5,
    legend=dict(
        title='Crime Type',
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    ),
    xaxis_title='Longitude',
    yaxis_title='Latitude'
)

fig_map.show()
print("      ✓ Interactive map complete - zoom into specific neighborhoods")

# ===== 2. INTERACTIVE HOURLY TREND =====
print("  [2/4] Creating interactive hourly trend analysis...")

fig_hourly = go.Figure()

# All crimes trace
fig_hourly.add_trace(go.Scatter(
    x=crimes_by_hour['hour'],
    y=crimes_by_hour['total_incidents'],
    mode='lines+markers',
    name='All Crimes',
    line=dict(color='#1976d2', width=4),
    marker=dict(size=10, symbol='circle'),
    hovertemplate='<b>Hour: %{x}:00</b><br>Total Incidents: %{y}<extra></extra>',
    fill='tozeroy',
    fillcolor='rgba(25, 118, 210, 0.1)'
))

# Violent crimes trace
fig_hourly.add_trace(go.Scatter(
    x=crimes_by_hour['hour'],
    y=crimes_by_hour['violent_incidents'],
    mode='lines+markers',
    name='Violent Crimes',
    line=dict(color='#d32f2f', width=4, dash='dash'),
    marker=dict(size=10, symbol='square'),
    hovertemplate='<b>Hour: %{x}:00</b><br>Violent Crimes: %{y}<extra></extra>'
))

# Add safe/dangerous zones
fig_hourly.add_vrect(
    x0=-0.5, x1=6.5,
    fillcolor="green", opacity=0.15,
    layer="below", line_width=0,
    annotation_text="✅ SAFE HOURS", 
    annotation_position="top left",
    annotation=dict(font_size=12, font_color="darkgreen")
)

fig_hourly.add_vrect(
    x0=17, x1=23.5,
    fillcolor="red", opacity=0.15,
    layer="below", line_width=0,
    annotation_text="⚠️ HIGH RISK HOURS", 
    annotation_position="top right",
    annotation=dict(font_size=12, font_color="darkred")
)

fig_hourly.update_layout(
    title='📈 Crime Trends Throughout the Day (Interactive)',
    xaxis_title='Hour of Day (0-23)',
    yaxis_title='Number of Incidents',
    height=550,
    hovermode='x unified',
    title_font_size=18,
    title_font_family='Arial',
    title_x=0.5,
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig_hourly.show()
print("      ✓ Hourly trend complete - plan your schedule around safe hours")

# ===== 3. NEIGHBORHOOD SAFETY RANKING =====
print("  [3/4] Creating interactive safety ranking...")

# Get top 15 highest-risk neighborhoods
top_15_safety = safety_by_neighborhood.nsmallest(15, 'safety_score').copy()

# Create custom hover text
top_15_safety['hover_info'] = (
    '<b>' + top_15_safety['neighborhood'] + '</b><br>' +
    'Safety Score: ' + top_15_safety['safety_score'].astype(str) + '<br>' +
    'Total Crimes: ' + top_15_safety['total_crimes'].astype(str) + '<br>' +
    'Violent Crimes: ' + top_15_safety['violent_crimes'].astype(str) + '<br>' +
    'Zip Code: ' + top_15_safety['zip_code'].astype(str) + '<br>' +
    'Risk Level: ' + top_15_safety['risk_level']
)

fig_safety = px.bar(
    top_15_safety,
    x='safety_score',
    y='neighborhood',
    color='risk_level',
    orientation='h',
    title='🛡️ Safety Scores: 15 Highest-Risk Neighborhoods',
    labels={'safety_score': 'Safety Score (0-100)', 'neighborhood': ''},
    color_discrete_map={
        'High Risk': '#d32f2f', 
        'Medium Risk': '#ff9800', 
        'Low Risk': '#4caf50'
    },
    hover_name='hover_info',
    hover_data={'safety_score': False, 'neighborhood': False, 'risk_level': False}
)

fig_safety.update_layout(
    height=650,
    title_font_size=18,
    title_font_family='Arial',
    title_x=0.5,
    yaxis={'categoryorder':'total ascending'},
    xaxis=dict(range=[0, 100]),
    showlegend=True,
    legend=dict(title='Risk Classification')
)

# Add reference lines
fig_safety.add_vline(x=50, line_dash="dash", line_color="red", 
                     annotation_text="High Risk Threshold", 
                     annotation_position="top right")
fig_safety.add_vline(x=70, line_dash="dash", line_color="orange", 
                     annotation_text="Medium Risk Threshold", 
                     annotation_position="bottom right")

fig_safety.show()
print("      ✓ Safety ranking complete - avoid red/orange neighborhoods")

# ===== 4. DAY OF WEEK ANALYSIS =====
print("  [4/4] Creating day-of-week comparison...")

crimes_by_day_chart = crimes_by_day.copy()
crimes_by_day_chart['is_weekend'] = crimes_by_day_chart['day'].isin(['Friday', 'Saturday', 'Sunday'])
crimes_by_day_chart['color_group'] = crimes_by_day_chart['is_weekend'].map({
    True: 'Weekend (Higher Risk)', 
    False: 'Weekday (Lower Risk)'
})

# Create custom hover
crimes_by_day_chart['hover_detail'] = (
   '<b>' + crimes_by_day_chart['day'].astype(str) + '</b><br>' +
    'Total Incidents: ' + crimes_by_day_chart['total_incidents'].astype(str) + '<br>' +
    'Violent Crimes: ' + crimes_by_day_chart['violent_incidents'].astype(str) + '<br>' +
    'Risk Level: ' + crimes_by_day_chart['color_group']
)

fig_days = px.bar(
    crimes_by_day_chart,
    x='day',
    y='total_incidents',
    color='color_group',
    title='📅 Crime Frequency by Day of Week',
    labels={'total_incidents': 'Total Incidents', 'day': ''},
    color_discrete_map={
        'Weekend (Higher Risk)': '#d32f2f', 
        'Weekday (Lower Risk)': '#4caf50'
    },
    hover_name='hover_detail',
    hover_data={'total_incidents': False, 'day': False, 'color_group': False}
)

fig_days.update_layout(
    height=550,
    title_font_size=18,
    title_font_family='Arial',
    title_x=0.5,
    showlegend=True,
    legend=dict(title='', orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Add median reference line
median_crimes = crimes_by_day_chart['total_incidents'].median()
fig_days.add_hline(y=median_crimes, line_dash="dash", line_color="gray",
                   annotation_text=f"Median ({median_crimes:.0f})",
                   annotation_position="right")

fig_days.show()
print("      ✓ Day-of-week analysis complete - weekdays are safer")

print("\n" + "="*80)
print("✅ INTERACTIVE VISUALIZATIONS COMPLETE")
print("="*80)
print("\n📊 Summary: 4 Interactive Charts Generated")
print("   1. Geographic Crime Map (identify hot spots)")
print("   2. Hourly Trend Analysis (plan daily schedule)")
print("   3. Neighborhood Safety Ranking (avoid high-risk areas)")
print("   4. Day-of-Week Patterns (plan weekly schedule)")
print("\n💡 All charts support:")
print("   • Hover for detailed information")
print("   • Zoom and pan controls")
print("   • Export as PNG images")
print("   • Legend filtering (click to hide/show)")
print("\n🎯 Use these charts together to build your optimal driving strategy!\n")

## Project Summary & Key Findings

---

### **Key Findings**

#### 1. **Temporal Patterns**
- **Safest Hours**: 4 AM - 6 AM (lowest crime activity)
- **Most Dangerous Hours**: 5 PM - Midnight (peak crime period)
- **Safer Days**: Monday, Tuesday, Thursday
- **Higher Risk Days**: Friday, Saturday, Sunday

#### 2. **Spatial Patterns**
- **Highest Crime Zip Codes**: 63104, 63103, 63101
- **Most Dangerous Neighborhoods**: Dutchtown, Central West End, Downtown
- **Safer Neighborhoods**: Soulard, Old North St. Louis, JeffVanderLou

#### 3. **Crime Categories**
- **Property Crimes**: ~37% (most common)
- **Society Crimes**: ~26%
- **Crimes Against Persons**: ~20% (most concerning for driver safety)

#### 4. **Zip Code 63102 Specific Insights**
- Total crime incidents analyzed: 78
- Violent crime incidents: 17
- **Highest Risk Area**: Columbus Square (11 violent crimes)
- **Safer Areas**: Carr Square, DeBaliviere Place, Soulard

---

### **Recommendation System Results**

#### For Uber Drivers:

**✅ RECOMMENDED Work Strategy:**
- Work during **morning hours (4-11 AM)**
- Focus on **weekdays (Mon-Thu)**
- Prioritize neighborhoods with **safety score > 70**
- Avoid areas with **any violent crime history**

**❌ AVOID:**
- Evening/night shifts **(5 PM - Midnight)** in high-risk areas
- **Columbus Square** and **Downtown** areas in Zip 63102
- **Weekend nights** in general

#### Optimal Locations (Highest Safety + Activity):
1. Forest Park Southeast (various hours)
2. Holly Hills (daytime)
3. Boulevard Heights (morning)
4. Fox Park (afternoon)
5. Botanical Heights (midday)

---

### **Data Quality & Limitations**

#### Strengths:
- **Large Dataset**: Thousands of crime records
- **Detailed Fields**: Location, time, crime type, severity
- **Geographic Coverage**: Multiple zip codes
- **Recent Data**: Up to February 2026

#### Limitations:
1. **Proxy for Demand**: Crime activity ≠ rider demand
   - High crime may actually reduce rider demand
   - Need actual ride request data for validation

2. **Temporal Coverage**: Single month (February 2026)
   - Seasonal variations not captured
   - Need multi-month/multi-year data for trends

3. **Zip Code Assignment**: Nearest-neighbor approach
   - May have boundary errors
   - More precise geocoding would improve accuracy

4. **Underreporting**: Not all crimes are reported
   - Actual safety may vary from recorded data

5. **No Socioeconomic Data**: Missing context
   - Income levels, population density, etc.
   - Would enrich understanding of patterns

---

### **Technical Achievements**

✓ **Data Integration**: Successfully merged API and CSV data  
✓ **Feature Engineering**: Extracted hour, day, month temporal features  
✓ **Scoring System**: Developed safety + activity combined metric  
✓ **Visualization**: Created 8+ static + 4 interactive charts  
✓ **Recommendation Engine**: Built actionable guidance system  
✓ **Documentation**: Comprehensive code comments and explanations  

---

### **Next Steps for Project Expansion**

#### Short Term (Next 2 Weeks):
1. **Data Enhancement**
   - Acquire multi-month crime data (at least 6-12 months)
   - Obtain actual Uber ride request data (if available)
   - Add demographic/socioeconomic data by neighborhood

2. **Model Improvement**
   - Implement machine learning for risk prediction
   - Add time-series forecasting for crime trends
   - Develop personalized recommendations based on driver preferences

3. **Validation**
   - Survey actual Uber drivers for feedback
   - Compare recommendations to their real-world experience
   - A/B test different recommendation strategies

#### Medium Term (Next Month):
4. **Feature Additions**
   - Real-time crime alert system
   - Route optimization (safest path between pickups)
   - Weather impact analysis
   - Event-based surge prediction

5. **Interactive Dashboard**
   - Build web application with Dash/Streamlit
   - Allow drivers to input their availability
   - Generate personalized schedules
   - Include real-time updates

6. **Mobile Application**
   - Develop mobile-friendly interface
   - Push notifications for alerts
   - GPS integration for current location safety

#### Long Term (Next Semester):
7. **Scalability**
   - Expand to other cities (Kansas City, Chicago, etc.)
   - Build automated data pipeline
   - Cloud deployment (AWS/Azure)

8. **Advanced Analytics**
   - Deep learning for pattern recognition
   - Natural language processing on incident descriptions
   - Computer vision for street safety assessment

9. **Business Model**
   - Pitch to Uber/Lyft as B2B product
   - Subscription model for independent drivers
   - Partnership with insurance companies

---






